In [1]:
import sqlite3

def update_experiment_view(db_path='../cem_results.db'):
    """
    Drops the existing experiment_summary view and recreates it 
    with the updated metrics schema including the testset column.
    """
    # Define the SQL script
    sql_script = """
    DROP VIEW IF EXISTS experiment_summary;

    CREATE VIEW experiment_summary AS
    SELECT 
        r.dataset,
        r.entity, 
        r.model_type,
        r.train_size, 
        r.seed, 
        r.lm,
        m.pollution, 
        m.iteration, 
        m.testset,
        m.f1_score,
        m.precision,
        m.recall,
        m.is_final,
        r.run_id,
        r.timestamp
    FROM metrics m
    JOIN runs r ON m.run_id = r.run_id;
    """

    try:
        # Establish connection
        conn = sqlite3.connect(db_path)
        cursor = conn.cursor()
        
        # Execute the script (handles multiple statements)
        cursor.executescript(sql_script)
        
        conn.commit()
        print("View 'experiment_summary' has been updated successfully.")
        
    except sqlite3.Error as e:
        print(f"An error occurred: {e}")
        
    finally:
        if conn:
            conn.close()

# Run the update


In [ ]:
import sqlite3
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# 1. Connection
conn = sqlite3.connect('../cem_results.db')
update_experiment_view()

target_entity = "track"
target_size = 1.0
target_seed = 1337
target_pollution = "high"


try:

    summary_query = f"""
        SELECT * FROM experiment_summary 
        WHERE entity = '{target_entity}' AND
        seed = {target_seed} AND
        train_size = '{target_size}' AND
        pollution = '{target_pollution}'
        ORDER BY "timestamp" DESC
    """
    
    # Execute and load into DataFrame
    df_results = pd.read_sql_query(summary_query, conn)
    display(df_results)

finally:
    conn.close()


results_filtered = df_results.drop(columns=["dataset", "model_type", "lm", "testset", "is_final", "run_id", "timestamp"])
print(results_filtered.to_csv())

View 'experiment_summary' has been updated successfully.


DatabaseError: Execution failed on sql '
        SELECT * FROM experiment_summary 
        WHERE entity = 'track' AND
        WHERE seed = 1337 AND
        train_size = '1.0' AND
        pollution = 'high'
        ORDER BY "timestamp" DESC
    ': near "WHERE": syntax error

In [3]:


import sqlite3
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# 1. Connection
conn = sqlite3.connect('../cem_results.db')
update_experiment_view()
i = 8 # Set your desired number of recent runs here

try:
    summary_query = f"""
        SELECT * FROM experiment_summary WHERE model_type = 'baseline' ORDER BY "timestamp"  desc LIMIT 20
    """
    
    # Execute and load into DataFrame
    df_baseline = pd.read_sql_query(summary_query, conn)
    
    print(f"Successfully retrieved data for {len(df_results)} runs.")
    baseline_filtered = df_baseline.drop(columns=["dataset", "lm", "testset", "is_final", "run_id", "timestamp"])
    display(baseline_filtered)

finally:
    conn.close()

# print(df_baseline.to_csv())

View 'experiment_summary' has been updated successfully.
Successfully retrieved data for 60 runs.


,entity,model_type,train_size,seed,pollution,iteration,f1_score,precision,recall
0,name,baseline,1.0,30,high,0,0.917,0.953,0.883
1,name,baseline,1.0,30,high,0,0.020,0.021,0.019
2,name,baseline,1.0,30,high,0,0.026,0.250,0.014
3,name,baseline,1.0,30,high,1,0.460,0.313,0.864
4,name,baseline,1.0,30,high,1,0.898,0.935,0.864
5,name,baseline,1.0,30,high,2,0.454,0.311,0.840
6,name,baseline,1.0,30,high,2,0.883,0.933,0.837
7,name,baseline,1.0,30,high,3,0.460,0.311,0.883
8,name,baseline,1.0,30,high,3,0.909,0.937,0.883
9,name,baseline,1.0,20,high,0,0.917,0.953,0.883


In [4]:
target_entity = "movie"
target_size = 1.0
target_seed = 42
target_pollution = "high"

# Filter the dataframe
mask = (
    # (results_filtered["entity"] == target_entity) &
    (results_filtered["train_size"] == target_size) &
    (results_filtered["seed"] == target_seed) &
    (results_filtered["pollution"] == target_pollution)
)

display(results_filtered[mask])

# Filter the dataframe
mask = (
    # (results_filtered["entity"] == target_entity) &
    (baseline_filtered["train_size"] == target_size) &
    (baseline_filtered["seed"] == target_seed) &
    (baseline_filtered["pollution"] == target_pollution)
)

display(baseline_filtered[mask])



import pandas as pd

# Define the columns to match on
match_cols = ["train_size", "seed", "pollution", "entity"]

# Perform an inner join to align the rows
df_comparison = pd.merge(
    df_results, 
    df_baseline, 
    on=match_cols, 
    suffixes=('_res', '_base')
)

display(df_comparison)
# Now you can calculate differences, for example:
df_comparison['score_diff'] = df_comparison['f1_score_res'] - df_comparison['f1_score_base']
display(df_comparison)

,entity,train_size,seed,pollution,iteration,f1_score,precision,recall
0,track,1.0,42,high,0,0.979,0.975,0.983
1,track,1.0,42,high,0,0.985,0.987,0.983
2,track,1.0,42,high,0,0.903,0.850,0.963
3,track,1.0,42,high,0,0.979,0.976,0.983
4,track,1.0,42,high,0,0.786,0.647,1.000
5,track,1.0,42,high,0,0.985,0.979,0.992
6,track,1.0,42,high,0,0.466,0.309,0.944
7,track,1.0,42,high,0,0.807,0.708,0.939
8,track,1.0,42,high,0,1.000,1.000,1.000
9,track,1.0,42,high,0,0.872,0.964,0.796


,entity,model_type,train_size,seed,pollution,iteration,f1_score,precision,recall


,dataset_res,entity,model_type_res,train_size,seed,lm_res,pollution,iteration_res,testset_res,f1_score_res,...,model_type_base,lm_base,iteration_base,testset_base,f1_score_base,precision_base,recall_base,is_final_base,run_id_base,timestamp_base


,dataset_res,entity,model_type_res,train_size,seed,lm_res,pollution,iteration_res,testset_res,f1_score_res,...,lm_base,iteration_base,testset_base,f1_score_base,precision_base,recall_base,is_final_base,run_id_base,timestamp_base,score_diff
